# Collect all data signals

Fetch all data signals since 2025-01-01. 
Combine them into one or multiple tables to share with Will.
Investments
All companies established or added to the database this year
(double check our criteria for Slack updates)
All new funding rounds from this year
Research projects
All research projects added or starting this year
(check how much signal we get if only use those that start this year)


We could also monitor new outputs of already running projects in future iterations
Policy
All debates and highlighted quotes from this year

In [3]:
from discovery_utils.getters import crunchbase
from discovery_utils import PROJECT_DIR, LOCAL_VECTOR_DB_PATH, logging

from typing import Literal

In [232]:
from discovery_utils.utils.llm.batch_check import LLMProcessor, generate_system_message

In [2]:
CB = crunchbase.CrunchbaseGetter()


2025-02-17 14:11:48,833 - discovery_utils.getters.crunchbase - INFO - Checking for latest version of data in S3 bucket: discovery-iss
2025-02-17 14:11:48,962 - discovery_utils.getters.crunchbase - INFO - Latest Crunchbase version found: Crunchbase_2025-02-17


In [105]:
import importlib
importlib.reload(crunchbase);

In [40]:
CB_ = crunchbase.CrunchbaseGetter()
CB_._organisations_enriched = CB._organisations_enriched
CB_._funding_rounds_enriched = CB._funding_rounds_enriched

2025-02-18 11:45:29,587 - discovery_utils.getters.crunchbase - INFO - Checking for latest version of data in S3 bucket: discovery-iss
2025-02-18 11:45:29,692 - discovery_utils.getters.crunchbase - INFO - Latest Crunchbase version found: Crunchbase_2025-02-17


In [13]:
from datetime import date
from dateutil.relativedelta import relativedelta

def define_dates(year: int, quarter: int, offset_start_date: int = 0) -> tuple[str, str]:
    """Get start and end dates for a given quarter of a year."""
    if quarter not in {1, 2, 3, 4}:
        raise ValueError("Quarter must be between 1 and 4.")

    # Determine the start month of the quarter
    start_month = (quarter - 1) * 3 + 1
    start_date = date(year, start_month, 1) - relativedelta(days=offset_start_date)
    
    # End date is the last day of the quarter
    end_date = start_date + relativedelta(months=3, days=-1)

    return start_date.isoformat(), end_date.isoformat()

In [29]:
mission = "ASF"
year = 2025
quarter = 1

start_date, end_date = define_dates(year=year, quarter=quarter, offset_start_date=1)

In [46]:
orgs = CB_.get_companies_in_nesta_categories("mission_labels", ["ASF"])

In [20]:
CB.funding_rounds_enriched.investment_type.unique()

array(['series_unknown', 'series_a', 'series_b', 'seed', 'pre_seed',
       'private_equity', 'grant', 'non_equity_assistance',
       'post_ipo_equity', 'series_c', 'debt_financing', 'series_d',
       'series_g', 'angel', 'convertible_note', 'undisclosed',
       'secondary_market', 'series_e', 'corporate_round', 'series_f',
       'post_ipo_debt', 'series_h', 'equity_crowdfunding',
       'post_ipo_secondary', 'product_crowdfunding', 'series_i',
       'initial_coin_offering', 'series_j'], dtype=object)

In [197]:
countries = (
    crunchbase.REGION_TO_COUNTRIES["UK"]
    + crunchbase.REGION_TO_COUNTRIES["Europe"]
    + crunchbase.REGION_TO_COUNTRIES["North America + Australia"]
)

In [280]:
cols_funding_rounds = [
    "funding_round_name", 
    "org_name",
    'mission_labels',
    'topic_labels',    
    "cb_url",
    "country_code", 
    "region_nesta",
    "region",
    "city",
    "year",
    "announced_on",
    "investment_type", 
    "investment_stage",
    "raised_amount_gbp",
    "raised_amount_usd",
    "raised_amount", 
    "raised_amount_currency_code",
    "post_money_valuation_usd", 
    "post_money_valuation",
    "post_money_valuation_currency_code", 
    "investor_count",
    "investor_name",
    "is_lead_investor",
    "investor_types",
]

In [281]:
cols_companies = [
    "name", 
    "short_description", 
    "founded_on", 
    'created_at',    
    "cb_url", 
    "homepage_url", 
    'mission_labels',
    'topic_labels',
    "rank", 
    "country_code", 
    "region_nesta",
    "region", 
    "city", 
    "status", 
    "category_list", 
    "closed_on", 
    "employee_count", 
    "email", 
    "phone", 
    "facebook_url", 
    "linkedin_url", 
    "twitter_url", 
    "logo_url", 
    "num_exits", 
    "num_funding_rounds", 
    "last_funding_on", 
    "investment_funding_gbp", 
    "num_investment_rounds", 
    "grant_funding_gbp", 
    "num_grants", 
    "total_funding_gbp", 
    "smart_money", 
]

In [282]:
rounds = (
    CB.funding_rounds_enriched
    .query("announced_on >= @start_date and announced_on <= @end_date")
    .query("country_code in @countries")
    .query("org_id in @orgs.id.to_list()")
    .assign(investment_category = lambda df: df.investment_type.map(crunchbase.investment_type_to_stage()))
)

In [283]:
def agg_list(x) -> str:
    return ", ".join([str(item) for item in list(x)])

rounds_investors = (
    rounds[['funding_round_id', 'investor_name', 'is_lead_investor', 'investor_types', 'investor_url']]
    .groupby('funding_round_id').agg(agg_list).reset_index()
)

_rounds = (
    rounds
    .drop(columns=['investor_name', 'is_lead_investor', 'investor_types', 'investor_url'], axis=1)
    .drop_duplicates("funding_round_id")
    .merge(rounds_investors, on='funding_round_id')
    .assign(investment_stage = lambda df: df.investment_type.map(crunchbase.investment_type_to_stage()))
    .merge(orgs[['id', 'mission_labels', 'topic_labels']].rename(columns={'id': 'org_id'}), on='org_id', how='left')
    .assign(region_nesta = lambda df: df.country_code.map(crunchbase.country_to_region()))
    .fillna("")
    .astype(str)
    .reset_index(drop=True)
)[cols_funding_rounds]


In [284]:
_rounds

,funding_round_name,org_name,mission_labels,topic_labels,cb_url,country_code,region_nesta,region,city,year,...,raised_amount_usd,raised_amount,raised_amount_currency_code,post_money_valuation_usd,post_money_valuation,post_money_valuation_currency_code,investor_count,investor_name,is_lead_investor,investor_types
0,Post-IPO Debt - Ensign Energy Services,Ensign Energy Services,ASF,Geothermal energy,https://www.crunchbase.com/funding_round/ensig...,CAN,North America + Australia,Alberta,Calgary,2024,...,17424.602,25000.0,CAD,,,,6.0,"N. Murray Edwards, Barth E. Whitham, Darlene J...","None, None, None, None, None, None","angel, angel, angel, angel, angel, angel"
1,Series A - geCKo Materials,geCKo Materials,"ASF,X","Data science & AI,Decarbonisation - General,Mo...",https://www.crunchbase.com/funding_round/gecko...,USA,North America + Australia,California,Los Gatos,2024,...,,,,,,,2.0,"KittyHawk, OpAmp Capital","True, None","venture_capital, venture_capital"
2,Debt Financing - Aypa Power,Aypa Power,ASF,"Renewables - General,Energy storage",https://www.crunchbase.com/funding_round/nrsto...,CAN,North America + Australia,Ontario,Toronto,2024,...,190000.0,190000.0,USD,,,,4.0,"Morgan Stanley Renewables, SMBC, Siemens Finan...","None, None, None, None","None, None, private_equity_firm, investment_bank"
3,Funding Round - bobbie,bobbie,"ASF,AFS,X","Data science & AI,Inclusion,Literacy,Decarboni...",https://www.crunchbase.com/funding_round/bobbi...,DEU,Europe,Bayern,Munich,2024,...,12937.481,12500.0,EUR,,,,,None,None,None
4,Series A - Circunomics,Circunomics,"ASF,X","Data science & AI,Mobile,Energy storage",https://www.crunchbase.com/funding_round/circu...,DEU,Europe,Rheinland-Pfalz,Mainz,2024,...,8279.988,8000.0,EUR,,,,4.0,"Green European Tech Fund, GG Rise, ORLEN VC, S...","True, None, None, True","venture_capital, None, corporate_venture_capit..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,Debt Financing - Hydrostor,Hydrostor,"ASF,X","Mobile,Energy storage",https://www.crunchbase.com/funding_round/hydro...,CAN,North America + Australia,Ontario,Toronto,2025,...,50000.0,50000.0,USD,,,,1.0,Canada Growth Fund Investment Management,True,None
146,Venture Round - Renew Risk,Renew Risk,"ASF,X","Training,Renewables - General",https://www.crunchbase.com/funding_round/renew...,GBR,UK,England,London,2025,...,6144.029,5900.0,EUR,,,,2.0,"Insurtech Gateway, Molten Ventures","None, True","angel_group,incubator,venture_capital, venture..."
147,Pre Seed Round - Elio,Elio,"ASF,X","Data science & AI,Decarbonisation - General",https://www.crunchbase.com/funding_round/elio-...,USA,North America + Australia,Delaware,Dover,2025,...,2000.0,2000.0,USD,,,,8.0,"Ananda Impact Ventures, Andreas Treichl, Overv...","True, None, None, None, None, None, None, None","venture_capital, angel, venture_capital, angel..."
148,Post-IPO Equity - EnviroGold Global,EnviroGold Global,"ASF,X","Data science & AI,Training,Decarbonisation - G...",https://www.crunchbase.com/funding_round/envir...,CAN,North America + Australia,Ontario,Toronto,2025,...,2802.427,4000.0,CAD,,,,,None,None,None


In [213]:
new_orgs = (
    orgs
    .query("(founded_on >= @start_date and founded_on <= @end_date) or (created_at >= @start_date and created_at <= @end_date)")
    .query("country_code in @countries")
    .assign(region_nesta = lambda df: df.country_code.map(crunchbase.country_to_region()))
    .sort_values("founded_on", ascending=False)
    .fillna("")
    .astype(str)
    .reset_index(drop=True)    
)[cols_companies]

In [214]:
from discovery_utils.utils import google
import os
import importlib
importlib.reload(os);
importlib.reload(google);

In [204]:
sheet_id = "1w2nSas1LwmPQY9HK-drrxIdGpDV3wU3pePDibHPVqL8"

In [285]:
google.upload_data_to_gsheet(sheet_id, {"crunchbase_funding": _rounds})
google.format_gsheet(sheet_id, "crunchbase_funding", freeze_cols=2)

2025-02-18 18:43:02,295 - root - INFO - Connected to Google Sheet: Mission Radar 2025 Q1: Test data [2025-02-18]
2025-02-18 18:43:04,415 - root - INFO - Uploading DataFrame to sheet: crunchbase_funding
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
2025-02-18 18:43:20,978 - root - INFO - Upload completed successfully.
2025-02-18 18:43:21,666 - root - INFO - Connected to Google Sheet: Mission Radar 2025 Q1: Test data [2025-02-18]


In [ ]:


google.upload_data_to_gsheet(sheet_id, {"crunchbase_companies": new_orgs})
google.format_gsheet(sheet_id, "crunchbase_companies", freeze_cols=4)

2025-02-18 17:53:27,888 - root - INFO - Connected to Google Sheet: test sheet
2025-02-18 17:53:30,142 - root - INFO - Uploading DataFrame to sheet: crunchbase_funding
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
2025-02-18 17:53:49,839 - root - INFO - Upload completed successfully.
2025-02-18 17:53:51,685 - root - INFO - Connected to Google Sheet: test sheet
2025-02-18 17:53:56,804 - root - INFO - Connected to Google Sheet: test sheet
2025-02-18 17:53:58,972 - root - INFO - Uploading DataFrame to sheet: crunchbase_companies
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.11/site-packages/df2gsprea

In [279]:
google.format_gsheet(sheet_id, "crunchbase_funding", freeze_cols=2)

2025-02-18 18:36:28,041 - root - INFO - Connected to Google Sheet: test sheet


## Gateway to Research

In [249]:
from datetime import datetime

def convert_to_date(x: str) -> str:
    try:
        return datetime.fromtimestamp(x / 1000).strftime('%Y-%m-%d')
    except:
        return ""

In [274]:
cols_projects = [
    'title',
    'is_relevant',       
    'mission_labels',
    'topic_labels',
    'status', 
    'grantCategory',
    'leadFunder',
    'abstractText',
    'techAbstractText',
    'potentialImpact',
    'start',
    'end',
    'amount',
    'url',
]

In [238]:
from discovery_utils.getters import gtr
from discovery_utils.utils import keywords as kw
import pandas as pd

In [217]:
GTR = gtr.GtrGetter()

2025-02-18 17:56:39,677 - discovery_utils.getters.gtr - INFO - Checking for latest version of data in S3 bucket: discovery-iss
2025-02-18 17:56:39,772 - discovery_utils.getters.gtr - INFO - Latest version found: GtR_20250216


In [255]:
new_projects = (
    GTR.projects_enriched
    .assign(created = lambda df: df.created.apply(convert_to_date))
    .query("(start >= @start_date and start <= @end_date)")
)
new_projects_text = GTR.get_projects_text().query("id in @new_projects.id.to_list()")

In [225]:
if len(new_projects) > 0:
    enrichment_df = kw.enrich_topic_labels(new_projects_text)

In [234]:
filename = f"llm_check.jsonl"
config_filename = f"config_{mission}.yaml"
system_message = generate_system_message(config_filename)
fields = [
    {"name": "is_relevant", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},
]

check_data = dict(zip(new_projects_text['id'], new_projects_text['text']))

In [ ]:
processor = LLMProcessor(
    output_path=filename,
    system_message=system_message,
    session_name="mission_radar",
    output_fields=fields,
)

processor.run(check_data, batch_size=15, sleep_time=0.5)

<Task pending name='Task-5' coro=<LLMProcessor.process_text_data() running at /Users/karlis.kanders/Code/discovery_utils/discovery_utils/utils/llm/batch_check.py:126>>

2025-02-18 18:20:45,324 - root - INFO - Processing batch 1/24
2025-02-18 18:20:49,840 - root - INFO - Processing batch 2/24
2025-02-18 18:20:52,837 - root - INFO - Processing batch 3/24
2025-02-18 18:20:54,396 - root - INFO - Processing batch 4/24
2025-02-18 18:20:56,276 - root - INFO - Processing batch 5/24
2025-02-18 18:20:57,705 - root - INFO - Processing batch 6/24
2025-02-18 18:20:59,583 - root - INFO - Processing batch 7/24
2025-02-18 18:21:03,114 - root - INFO - Processing batch 8/24
2025-02-18 18:21:05,322 - root - INFO - Processing batch 9/24
2025-02-18 18:21:10,217 - root - INFO - Processing batch 10/24
2025-02-18 18:21:12,079 - root - INFO - Processing batch 11/24
2025-02-18 18:21:13,986 - root - INFO - Processing batch 12/24
2025-02-18 18:21:15,901 - root - INFO - Processing batch 13/24
2025-02-18 18:21:20,315 - root - INFO - Processing batch 14/24
2025-02-18 18:21:22,291 - root - INFO - Processing batch 15/24
2025-02-18 18:21:25,203 - root - INFO - Processing batch 16/24
2

In [239]:
relevant_check_df = pd.read_json(filename, lines=True)

In [275]:
_new_projects = (
    new_projects
    .merge(enrichment_df, on='id', how='left')
    .assign(mission_labels = lambda df: df.mission_labels.apply(lambda x: x.split(",") if (type(x) is str) else []))
    .explode('mission_labels')
    .query("mission_labels in @mission")
    .merge(relevant_check_df, left_on='id', right_on='id', how='left')
    .sort_values(["is_relevant", "start"], ascending=False)
    .drop_duplicates("id")
    .fillna("")
    .astype(str)
    .reset_index(drop=True)    
)[cols_projects]

In [276]:
_new_projects

,title,is_relevant,mission_labels,topic_labels,status,grantCategory,leadFunder,abstractText,techAbstractText,potentialImpact,start,end,amount,url
0,Generating Further Experimental Datasets for H...,yes,ASF,"Decarbonisation - General,Training,Data scienc...",Active,Collaborative R&D,Innovate UK,"In the current landscape, the scale of hydroge...",,,2025-01-01,2025-03-31,25019.0,https://gtr.ukri.org/projects?ref=10134842
1,LAB TEST - LAtent heat Battery TEsting for Sim...,yes,ASF,"Energy efficiency,Energy storage,Training,Data...",Active,Collaborative R&D,Innovate UK,Sunamp designs and manufactures space-saving t...,,,2025-01-01,2025-04-30,49040.0,https://gtr.ukri.org/projects?ref=10139837
2,36v low surface heat infrared heating system S...,yes,ASF,"Data science & AI,Mobile,Heat pumps,Energy eff...",Active,Collaborative R&D,Innovate UK,"The Energy Carbon 36v, low energy, far infrare...",,,2025-01-01,2025-12-31,48607.0,https://gtr.ukri.org/projects?ref=10139710
3,Testing tepeo ZEB in the Salford Energy House 1,yes,ASF,"Energy efficiency,Training,Renewables - Genera...",Active,Collaborative R&D,Innovate UK,Our project focuses on testing the Zero Emissi...,,,2025-01-01,2025-05-31,49951.0,https://gtr.ukri.org/projects?ref=10140951
4,"An innovative, circular, waste-repurposing ins...",yes,ASF,"Data science & AI,Energy efficiency",Active,Collaborative R&D,Innovate UK,The construction and agriculture industries ar...,,,2025-01-01,2025-12-31,344482.0,https://gtr.ukri.org/projects?ref=10138490
5,SAP Listing of ICAX’s Seren 10 Heat Pump,yes,ASF,"Energy efficiency,Training,Heat pumps,Data sci...",Active,Collaborative R&D,Innovate UK,**Project Aim:**\n\nThe aim of this project is...,,,2025-01-01,2025-08-31,49865.0,https://gtr.ukri.org/projects?ref=10140420
6,ORGANIC HETEROJUNCTION PHOTOCATALYSTS FOR SOLA...,no,ASF,"Recruitment,Data science & AI,Hydrogen energy,...",Active,Research and Innovation,EPSRC,The sustainable generation of green hydrogen i...,,,2025-03-01,2027-02-28,766886.0,https://gtr.ukri.org/projects?ref=EP/Z536258/1
7,Sustainable crop control: efficacy of bioinsec...,no,ASF,"Decarbonisation - General,Training,Data scienc...",Active,Collaborative R&D,Innovate UK,BugBiome has **harnessed crop-associated micro...,,,2025-03-01,2026-08-31,296160.0,https://gtr.ukri.org/projects?ref=10131210
8,Enhancing crops with C2 photosynthesis,no,ASF,"Bioenergy,Inclusion,Mobile,Data science & AI",Active,Fellowship,UKRI FLF,Our food security is at risk. Within the next ...,,,2025-02-01,2028-01-31,595578.0,https://gtr.ukri.org/projects?ref=MR/Z000424/1
9,The BLOOM (Co-Benefits of Largescale Organic F...,no,ASF,"Weight,Decarbonisation - General,Training,Data...",Active,Fellowship,UKRI FLF,"In December 2023, at COP28 in Dubai, the Food ...",,,2025-02-01,2028-01-31,535725.0,https://gtr.ukri.org/projects?ref=MR/Z000041/1


In [ ]:
google.upload_data_to_gsheet(sheet_id, {"ukri_projects": _new_projects})

2025-02-18 18:35:38,840 - root - INFO - Connected to Google Sheet: test sheet
2025-02-18 18:35:42,805 - root - INFO - Uploading DataFrame to sheet: ukri_projects
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
2025-02-18 18:35:56,485 - root - INFO - Upload completed successfully.
2025-02-18 18:35:57,556 - root - INFO - Connected to Google Sheet: test sheet


In [278]:
google.format_gsheet(sheet_id, "ukri_projects", freeze_cols=4)

2025-02-18 18:36:19,030 - root - INFO - Connected to Google Sheet: test sheet
